# Detecção de dias atípicos com Isolation Forest

- Name: Renan Santos Mendes
- Email: renansantosmendes@gmail.com
- Course subject: `<COURSE_SUBJECT_PLACEHOLDER>`
- Program: `<PROGRAM_PLACEHOLDER>`

Este notebook treina um detector de dias atípicos para o preço de um ativo.
O modelo é um Isolation Forest (scikit-learn) que aprende o padrão dos
retornos, do volume e de dois indicadores técnicos, e marca os dias que
fogem desse padrão. A avaliação é walk-forward: cada dia é pontuado só com
um modelo treinado nos dias anteriores, sem olhar o futuro.

## 1. Bibliotecas

A célula abaixo importa as bibliotecas usadas no notebook. Para instalar as
dependências do projeto, rode `uv sync --extra dev --extra notebook` na
raiz do repositório.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

## 2. Dados de preço

A função `load_prices` baixa o histórico do ativo no Yahoo Finance, com os
preços ajustados por proventos e desdobramentos. Ela devolve apenas as
colunas de fechamento e de volume.

In [ ]:
def load_prices(
    ticker: str,
    period: str = "3y",
) -> pd.DataFrame:
    """Download the adjusted close and volume of a ticker.

    Parameters
    ----------
    ticker : str
        Yahoo Finance symbol, for example ``PETR4.SA`` or ``AAPL``.
    period : str, optional
        History period accepted by yfinance. The default is ``3y``.

    Returns
    -------
    pd.DataFrame
        Frame indexed by date with the columns ``Close`` and ``Volume``,
        without missing values.

    Example
    -------
    >>> prices = load_prices("AAPL", period="1y")  # doctest: +SKIP
    """
    prices = yf.download(
        ticker,
        period=period,
        auto_adjust=True,
        progress=False,
    )
    if isinstance(prices.columns, pd.MultiIndex):
        prices.columns = prices.columns.get_level_values(0)
    return prices[["Close", "Volume"]].dropna()

## 3. Features

A função `build_features` transforma os preços em quatro entradas do modelo:

- **log_return**: retorno logarítmico diário.
- **volume_z_score**: quão atípico é o volume do dia em relação aos últimos
  60 dias.
- **rsi**: índice de força relativa de 14 dias, que mede o ritmo de altas
  contra baixas.
- **macd_normalized**: histograma do MACD dividido pelo preço, para comparar
  ativos de preços diferentes.

As funções `relative_strength_index` e `macd_histogram` calculam os
indicadores técnicos.

In [ ]:
def relative_strength_index(
    close: pd.Series,
    window: int = 14,
) -> pd.Series:
    """Compute the relative strength index of a close series.

    Parameters
    ----------
    close : pd.Series
        Daily closing prices.
    window : int, optional
        Number of daily changes in the rolling averages. The default is 14.

    Returns
    -------
    pd.Series
        RSI values on the same index as ``close``, from 0 to 100. The first
        ``window`` values are NaN.
    """
    daily_change = close.diff()
    average_gain = daily_change.clip(lower=0).rolling(window).mean()
    average_loss = (-daily_change.clip(upper=0)).rolling(window).mean()
    return 100 - 100 / (1 + average_gain / average_loss)


def macd_histogram(
    close: pd.Series,
    fast_span: int = 12,
    slow_span: int = 26,
    signal_span: int = 9,
) -> pd.Series:
    """Compute the MACD histogram of a close series.

    Parameters
    ----------
    close : pd.Series
        Daily closing prices.
    fast_span : int, optional
        Span of the fast exponential moving average. The default is 12.
    slow_span : int, optional
        Span of the slow exponential moving average. The default is 26.
    signal_span : int, optional
        Span of the signal line. The default is 9.

    Returns
    -------
    pd.Series
        Histogram values on the same index as ``close``: the MACD line minus
        its signal line.
    """
    macd_line = (
        close.ewm(span=fast_span, adjust=False).mean()
        - close.ewm(span=slow_span, adjust=False).mean()
    )
    signal_line = macd_line.ewm(span=signal_span, adjust=False).mean()
    return macd_line - signal_line


def build_features(prices: pd.DataFrame) -> pd.DataFrame:
    """Turn daily prices into the inputs of the detector.

    Parameters
    ----------
    prices : pd.DataFrame
        Frame with the columns ``Close`` and ``Volume``, sorted by date.

    Returns
    -------
    pd.DataFrame
        Frame with the columns ``log_return``, ``volume_z_score``, ``rsi``
        and ``macd_normalized``, without missing values. The first rows are
        removed because they do not have a full 60-day window.

    Example
    -------
    >>> build_features(load_prices("AAPL", period="1y"))  # doctest: +SKIP
    """
    close = prices["Close"]
    log_volume = np.log1p(prices["Volume"])
    features = pd.DataFrame(index=prices.index)
    features["log_return"] = np.log(close).diff()
    features["volume_z_score"] = (
        log_volume - log_volume.rolling(60).mean()
    ) / log_volume.rolling(60).std()
    features["rsi"] = relative_strength_index(close)
    features["macd_normalized"] = macd_histogram(close) / close
    return features.dropna()

## 4. Avaliação walk-forward

A função `walk_forward_scores` percorre o histórico em blocos de 21 dias. Em
cada bloco, ela treina um Isolation Forest com as 252 linhas anteriores
(cerca de um ano de pregões), escala as features com `RobustScaler` e pontua
os dias do bloco. Um score negativo marca o dia como atípico.

In [ ]:
def walk_forward_scores(
    features: pd.DataFrame,
    train_window: int = 252,
    refit_every: int = 21,
    contamination: float = 0.02,
    random_seed: int = 42,
) -> pd.Series:
    """Score each day with a model fitted only on the days before it.

    Parameters
    ----------
    features : pd.DataFrame
        Feature rows sorted by date, without missing values.
    train_window : int, optional
        Number of rows in each training window. The default is 252.
    refit_every : int, optional
        Number of rows scored before the model is refitted. The default is 21.
    contamination : float, optional
        Expected share of atypical rows. The default is 0.02.
    random_seed : int, optional
        Seed of the Isolation Forest. The default is 42.

    Returns
    -------
    pd.Series
        Scores of the rows after the first ``train_window`` rows. Negative
        scores mark atypical days.

    Example
    -------
    >>> walk_forward_scores(build_features(prices))  # doctest: +SKIP
    """
    scores = pd.Series(np.nan, index=features.index)
    for block_start in range(train_window, len(features), refit_every):
        block_end = min(block_start + refit_every, len(features))
        training_rows = features.iloc[block_start - train_window : block_start]
        block_rows = features.iloc[block_start:block_end]

        scaler = RobustScaler().fit(training_rows)
        model = IsolationForest(
            n_estimators=200,
            contamination=contamination,
            random_state=random_seed,
        )
        model.fit(scaler.transform(training_rows))
        scores.iloc[block_start:block_end] = model.decision_function(
            scaler.transform(block_rows)
        )
    return scores.dropna()

## 5. Treino e pontuação do ativo

A célula abaixo baixa os preços de `TICKER`, calcula as features e pontua
todos os dias com a avaliação walk-forward. Troque o valor de `TICKER` para
analisar outro ativo, como `AAPL`.

In [ ]:
TICKER = "PETR4.SA"

prices = load_prices(TICKER, period="3y")
features = build_features(prices)
scores = walk_forward_scores(features)

print(f"Linhas de preço: {len(prices)}")
print(f"Linhas com features: {len(features)}")
print(f"Dias pontuados: {len(scores)}")
print(f"Dias atípicos: {(scores < 0).sum()} ({(scores < 0).mean():.1%})")

## 6. Visualização

A função `plot_outliers` monta três painéis: o preço com os dias atípicos
em vermelho e os cinco mais extremos anotados, os retornos diários e o score
de anomalia com o limiar em zero. A figura é salva como PNG na pasta de
execução do notebook.

In [ ]:
def plot_outliers(
    prices: pd.DataFrame,
    scores: pd.Series,
    ticker: str,
    top_n: int = 5,
) -> None:
    """Plot the price, the returns and the scores with the atypical days.

    Parameters
    ----------
    prices : pd.DataFrame
        Frame with the column ``Close``, indexed by date.
    scores : pd.Series
        Walk-forward scores. Negative scores mark atypical days.
    ticker : str
        Ticker shown in the title and used in the file name.
    top_n : int, optional
        Number of most extreme days annotated in the price panel. The
        default is 5.

    Returns
    -------
    None
        The figure is shown and saved as ``<ticker>_outliers.png``.
    """
    close = prices["Close"].reindex(scores.index)
    log_ret = np.log(prices["Close"]).diff().reindex(scores.index)
    pct_ret = np.expm1(log_ret)
    flagged = (scores < 0).values
    top_dates = scores[flagged].nsmallest(top_n).index

    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "axes.spines.top": False,
        "axes.spines.right": False,
    })
    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, figsize=(12, 10), sharex=True,
        gridspec_kw={"height_ratios": [2, 1.1, 1.1]},
    )

    ax1.plot(close.index, close.values, color="#1f3b5c", lw=1.4, label="Fechamento")
    ax1.scatter(close.index[flagged], close.values[flagged],
                color="#d62728", s=34, zorder=3, label="Dia atípico")
    offsets = [(0, 22), (-38, 40), (38, 40), (-70, 58), (70, 58)]
    ax1.set_ylim(close.min() * 0.95, close.max() * 1.18)
    for position, day in enumerate(top_dates):
        x_offset, y_offset = offsets[position % len(offsets)]
        ax1.annotate(f"{day:%d/%m/%Y}\n{pct_ret[day]:+.1%}",
                     xy=(day, close[day]), xytext=(x_offset, y_offset),
                     textcoords="offset points",
                     ha="center", fontsize=8,
                     bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85),
                     arrowprops=dict(arrowstyle="-", color="gray", lw=0.7))
    ax1.set_title(f"{ticker}: dias atípicos detectados com IsolationForest",
                  loc="left", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Preço")
    ax1.legend(frameon=False, loc="upper left")

    colors = np.where(flagged, "#d62728", "#9fb3c8")
    ax2.bar(log_ret.index, log_ret.values, color=colors, width=1.0)
    ax2.axhline(0, color="black", lw=0.6)
    ax2.set_ylabel("Retorno log")

    ax3.plot(scores.index, scores.values, color="#1f3b5c", lw=1)
    ax3.axhline(0, color="#d62728", ls="--", lw=1, label="Limiar (score = 0)")
    ax3.fill_between(scores.index, scores.values, 0, where=flagged,
                     color="#d62728", alpha=0.25)
    ax3.set_ylabel("Score")
    ax3.set_xlabel("Data")
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
    ax3.legend(frameon=False, loc="lower left")

    for ax in (ax1, ax2, ax3):
        ax.grid(alpha=0.25)

    fig.text(0.01, 0.005,
             "Score < 0 = dia mais isolado do que o padrão dos 252 dias anteriores "
             "(ajuste walk-forward, sem dados futuros).",
             fontsize=8, color="gray")
    fig.tight_layout(rect=(0, 0.02, 1, 1))

    out_file = f"{ticker.replace('.', '_')}_outliers.png"
    fig.savefig(out_file, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Figura salva em {out_file}")

A célula abaixo desenha a figura para o ativo escolhido em `TICKER`.

In [ ]:
plot_outliers(prices, scores, TICKER)